[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weilinear/rsi_labs/blob/main/gemini_self_improvement.ipynb)

# Recursive Self-Improvement, Part 0: What Exactly Is Improving?

A frozen Gemini model proposes structured edits;
Python executes invoices and independently checks the results. This makes the
persistence and recursive boundary visible without generated-code execution.

We demonstrate output refinement, persistent agent improvement, and editing
the improvement policy that controls subsequent proposals. The editable space
is finite and deliberately simple. This is not model training or open-ended
algorithm discovery. The task agent is a configurable Python program; Gemini
is its proposer/improver, not the invoice execution engine.

**Setup:** Python 3.10+, `GEMINI_API_KEY` (preferred) or `GOOGLE_API_KEY` in the
kernel's environment. Optional `GEMINI_MODEL`; default `gemini-3.8-flash`.
The notebook uses the standard library and calls Google's generateContent API.
Its synthetic traces, schemas, and proposals are sent to Google. Keys are read
at runtime and never printed or included in artifacts. Rerunning live cells
makes API calls; usage metadata is recorded, but dollar cost is not estimated.

For harness checks without network, set `TOY_BACKEND=replay` before starting
Python. Replay uses a hand-authored fixture and is explicitly not Gemini.


In [1]:
from dataclasses import dataclass, asdict
from decimal import Decimal
from pathlib import Path
import copy
import hashlib
import json
import os
import re
import time
import urllib.error
import urllib.request

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
BACKEND = os.environ.get("TOY_BACKEND", "gemini")
ARTIFACT_DIR = Path("gemini_artifacts" if BACKEND == "gemini" else "replay_artifacts")
print("Backend:", BACKEND, "| Model:", MODEL if BACKEND == "gemini" else "fixture")


class GeminiJSON:
    def __init__(self, model=MODEL, max_calls=24):
        self.key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        if not self.key:
            raise RuntimeError("Set GEMINI_API_KEY or GOOGLE_API_KEY in the kernel environment.")
        if not re.fullmatch(r"[A-Za-z0-9._-]+", model):
            raise ValueError("GEMINI_MODEL must be a model ID, not a URL.")
        self.model, self.max_calls = model, max_calls
        self.cache, self.calls = {}, []

    def generate(self, context, schema):
        prompt = (
            "You propose changes for a small optimization experiment. Return JSON only. "
            "Task invoices obey unit_price * quantity * (1 - discount) + shipping. "
            "The evaluator and acceptance rule are fixed. Do not claim an edit is accepted. "
            "Use the current improvement policy to propose one candidate. "
            "There is ONE candidate evaluation per task adaptation episode. "
            "For an improver target, maximize the quality of the agents it produces. "
            "Respect the maximum changed-fields constraint for the CURRENT policy. "
            "The candidate contains the complete parameter object, preserving unchanged fields. "
            "You may return zero candidates if there is no useful change.\n"
            + json.dumps(context, sort_keys=True)
        )
        cache_key = hashlib.sha256((prompt + json.dumps(schema, sort_keys=True)).encode()).hexdigest()
        if cache_key in self.cache:
            return copy.deepcopy(self.cache[cache_key])
        if len(self.calls) >= self.max_calls:
            raise RuntimeError("Gemini call budget exhausted; no automatic retries.")
        payload = {
            "contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0, "maxOutputTokens": 4096,
                                 "responseMimeType": "application/json",
                                 "responseJsonSchema": schema},
        }
        request = urllib.request.Request(
            f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent",
            data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json", "x-goog-api-key": self.key},
            method="POST",
        )
        started = time.monotonic()
        try:
            with urllib.request.urlopen(request, timeout=45) as response:
                raw = json.load(response)
        except urllib.error.HTTPError as error:
            raise RuntimeError(f"Gemini returned HTTP {error.code}; no fallback or automatic retry.") from None
        except (urllib.error.URLError, TimeoutError):
            raise RuntimeError("Gemini network request failed or timed out; no fallback.") from None
        candidates = raw.get("candidates", [])
        if not candidates or candidates[0].get("finishReason") != "STOP":
            raise RuntimeError("Gemini returned no complete candidate; inspect model/output limits.")
        parts = candidates[0].get("content", {}).get("parts", [])
        output = "".join(p.get("text", "") for p in parts if not p.get("thought"))
        try:
            result = json.loads(output)
        except json.JSONDecodeError:
            raise RuntimeError("Gemini returned invalid JSON; no edit applied.") from None
        self.calls.append({"model": self.model, "model_version": raw.get("modelVersion"),
                           "seconds": round(time.monotonic() - started, 3),
                           "usage": raw.get("usageMetadata", {}),
                           "context": context, "response": result})
        self.cache[cache_key] = result
        return copy.deepcopy(result)


class ReplayJSON:
    """Explicit fixture for harness checks; never presented as model output."""
    def __init__(self):
        self.calls = []

    def generate(self, context, schema):
        params = copy.deepcopy(context["state"]["parameters"])
        limit = context["current_improvement_policy"]["max_fields_per_candidate"]
        if context["state"]["kind"] == "task":
            inputs = [trace["input"] for trace in context["feedback"].get("traces", [])]
            active = {
                "use_quantity": any(t["quantity"] != 1 for t in inputs),
                "use_discount": any(Decimal(t["discount"]) != 0 for t in inputs),
                "use_shipping": any(Decimal(t["shipping"]) != 0 for t in inputs),
            }
            missing = [key for key, value in params.items() if not value and active[key]]
            for key in missing[:limit]:
                params[key] = True
        else:
            params["max_fields_per_candidate"] = 3
        result = {"candidates": [params], "rationale": "Hand-authored replay fixture."}
        self.calls.append({"backend": "replay", "context": context, "response": result})
        return result


if BACKEND not in ("gemini", "replay"):
    raise ValueError("TOY_BACKEND must be gemini or replay")
client = GeminiJSON() if BACKEND == "gemini" else ReplayJSON()


Backend: gemini | Model: gemini-3.8-flash


## The fixed task agent and independent evaluator

Persistent task parameters are three Boolean rules. Gemini cannot change the
arithmetic interpreter, objective, holdout, or acceptance code. Its proposals
are parsed as data. No model-produced string is executed as Python.

Unlike the original deterministic notebook, feedback does not supply ready-made
patches. It contains invoice inputs, actual answers, and reference answers.
The proposer must decide which allowed parameters to change.


In [2]:
@dataclass(frozen=True)
class Invoice:
    unit_price: str
    quantity: int = 1
    discount: str = "0"
    shipping: str = "0"


def run_task(task, state):
    p = state["parameters"]
    total = Decimal(task.unit_price) * (task.quantity if p["use_quantity"] else 1)
    if p["use_discount"]:
        total *= 1 - Decimal(task.discount)
    if p["use_shipping"]:
        total += Decimal(task.shipping)
    return total.quantize(Decimal("0.01"))


def reference(task):
    return (Decimal(task.unit_price) * task.quantity * (1 - Decimal(task.discount))
            + Decimal(task.shipping)).quantize(Decimal("0.01"))


def evaluate_task(state, tasks):
    traces = [{"input": asdict(t), "actual": str(run_task(t, state)),
               "expected": str(reference(t)), "passed": run_task(t, state) == reference(t)}
              for t in tasks]
    return {"score": sum(t["passed"] for t in traces) / len(traces), "traces": traces}


INITIAL_AGENT = {"kind": "task", "parameters": {
    "use_quantity": False, "use_discount": False, "use_shipping": False}}
INITIAL_POLICY = {"kind": "improver", "parameters": {
    "max_fields_per_candidate": 1,
    "proposal_instruction": (
        "Inspect failures and propose changes that improve the measured objective. "
        "Use execution evidence. Consider the candidate evaluation budget."
    )}}
TASK_PROPERTIES = {key: {"type": "boolean"} for key in INITIAL_AGENT["parameters"]}
POLICY_PROPERTIES = {
    "max_fields_per_candidate": {"type": "integer", "minimum": 1, "maximum": 3},
    "proposal_instruction": {"type": "string", "minLength": 1, "maxLength": 1500},
}


def proposal_schema(kind):
    properties = TASK_PROPERTIES if kind == "task" else POLICY_PROPERTIES
    return {"type": "object", "properties": {
        "candidates": {"type": "array", "maxItems": 1, "items": {
            "type": "object", "properties": properties, "required": list(properties),
            "additionalProperties": False}},
        "rationale": {"type": "string"}},
        "required": ["candidates", "rationale"], "additionalProperties": False}


def validate_candidate(state, candidate, current_policy):
    original = state["parameters"]
    if not isinstance(candidate, dict) or set(candidate) != set(original):
        raise ValueError("Candidate must contain exactly the allowed fields.")
    if state["kind"] == "task":
        if any(type(value) is not bool for value in candidate.values()):
            raise ValueError("Task rules must be Boolean.")
    else:
        limit = candidate["max_fields_per_candidate"]
        text = candidate["proposal_instruction"]
        if type(limit) is not int or not 1 <= limit <= 3:
            raise ValueError("Candidate edit limit must be an integer between 1 and 3.")
        if not isinstance(text, str) or not text.strip() or len(text) > 1500:
            raise ValueError("Candidate instruction must contain 1–1500 characters.")
    changed = sum(candidate[key] != original[key] for key in original)
    if changed > current_policy["parameters"]["max_fields_per_candidate"]:
        raise ValueError("Candidate exceeds the CURRENT policy's edit limit.")
    return {"kind": state["kind"], "parameters": copy.deepcopy(candidate)}


def propose(state, feedback, policy, client):
    context = {"state": state, "feedback": feedback,
               "current_improvement_policy": policy["parameters"],
               "allowed_parameters": proposal_schema(state["kind"])["properties"]["candidates"]["items"],
               "candidate_evaluation_budget": 1}
    result = client.generate(context, proposal_schema(state["kind"]))
    if not isinstance(result, dict) or set(result) != {"candidates", "rationale"}:
        raise ValueError("Invalid proposal envelope.")
    if not isinstance(result["candidates"], list) or len(result["candidates"]) > 1:
        raise ValueError("At most one candidate per proposal call.")
    if not isinstance(result["rationale"], str):
        raise ValueError("Rationale must be a string.")
    candidates, rejected = [], []
    for item in result["candidates"]:
        try:
            candidates.append(validate_candidate(state, item, policy))
        except ValueError as error:
            rejected.append(str(error))
    return candidates, {"rationale": result["rationale"], "rejected": rejected}


def improve(state, evaluate, policy, client, budget=1):
    if type(budget) is not int or budget < 0:
        raise ValueError("Candidate budget must be a nonnegative integer.")
    best = copy.deepcopy(state)
    feedback = evaluate(best)
    log = {"baseline_score": feedback["score"], "candidate_evaluations": 0, "steps": []}
    while log["candidate_evaluations"] < budget and feedback["score"] < 1:
        candidates, proposal_log = propose(best, feedback, policy, client)
        if not candidates:
            log["steps"].append({"proposal": proposal_log, "accepted": False})
            break
        candidate = candidates[0]
        candidate_feedback = evaluate(candidate)
        log["candidate_evaluations"] += 1
        accepted = candidate_feedback["score"] > feedback["score"]
        log["steps"].append({"candidate": candidate, "score": candidate_feedback["score"],
                             "accepted": accepted, "proposal": proposal_log})
        if not accepted:
            break
        best, feedback = candidate, candidate_feedback
    log["final_score"] = feedback["score"]
    return best, log


DEVELOPMENT = [Invoice("10", quantity=3), Invoice("100", discount="0.20"),
               Invoice("20", shipping="5")]
HELD_OUT = [Invoice("7.50", quantity=4), Invoice("60", discount="0.25"),
            Invoice("13", shipping="2.75"),
            Invoice("12.50", quantity=3, discount="0.10", shipping="4")]


## 1. Output refinement: use feedback, retain nothing

Gemini sees one failed invoice and proposes a temporary rule change. We compute
a revised answer with that change, then discard it. The persistent agent stays
unchanged. A successful refinement need not make the next request better.


In [3]:
refinement_task = Invoice("100", discount="0.20")
refinement_feedback = evaluate_task(INITIAL_AGENT, [refinement_task])
temporary_candidates, refinement_log = propose(INITIAL_AGENT, refinement_feedback,
                                               INITIAL_POLICY, client)
temporary_agent = temporary_candidates[0] if temporary_candidates else copy.deepcopy(INITIAL_AGENT)
refinement_result = {
    "first_answer": str(run_task(refinement_task, INITIAL_AGENT)),
    "refined_answer": str(run_task(refinement_task, temporary_agent)),
    "next_request_answer": str(run_task(refinement_task, INITIAL_AGENT)),
    "passed": run_task(refinement_task, temporary_agent) == reference(refinement_task),
}
print(json.dumps(refinement_result, indent=2))
print("Gemini rationale:" if BACKEND == "gemini" else "Fixture rationale:", refinement_log["rationale"])
assert INITIAL_AGENT["parameters"] == {key: False for key in TASK_PROPERTIES}


{
  "first_answer": "100.00",
  "refined_answer": "80.00",
  "next_request_answer": "100.00",
  "passed": true
}
Gemini rationale: The feedback shows an expected value of 80.00 from a unit price of 100 with a 0.20 discount, but actual was 100.00 because discount was ignored. Setting use_discount to true changes 1 field within policy limits to correctly apply discounts.


## 2. Agent self-improvement: retain task parameters

Use one candidate evaluation on the three development invoices. Save and reload
an accepted agent. The current policy allows only one field change per candidate.
The held-out invoices are scored afterwards, never supplied to this proposal.


In [4]:
task_evaluator = lambda state: evaluate_task(state, DEVELOPMENT)
learned_agent, agent_log = improve(INITIAL_AGENT, task_evaluator, INITIAL_POLICY, client)
ARTIFACT_DIR.mkdir(exist_ok=True)
(ARTIFACT_DIR / "agent.json").write_text(json.dumps(learned_agent, indent=2))
learned_agent = json.loads((ARTIFACT_DIR / "agent.json").read_text())
print("Retained task parameters:", learned_agent["parameters"])
print("Development score:", task_evaluator(learned_agent)["score"])
print("Held-out score:", evaluate_task(learned_agent, HELD_OUT)["score"])
print("Accepted:", any(step["accepted"] for step in agent_log["steps"]))


Retained task parameters: {'use_quantity': True, 'use_discount': False, 'use_shipping': False}
Development score: 0.3333333333333333
Held-out score: 0.25
Accepted: True


## 3. Recursive self-improvement: optimize the improvement policy

Each candidate policy is evaluated by starting fresh task agents, letting it
improve them with one candidate evaluation, then scoring the resulting agents
on episode validation invoices. This tests how well the policy produces future
task improvements. The fixed oracle does not change.

The same `improve` function targets the current policy, using that policy to
propose its own replacement. The edit limit and proposal instructions are both
editable. The replacement is saved and reused below. Evaluation feedback includes
adaptation traces; it does not explicitly suggest a particular replacement.


In [5]:
META_DEVELOPMENT = [
    ([Invoice("8", quantity=2), Invoice("50", discount="0.10"), Invoice("15", shipping="3")],
     [Invoice("11", quantity=5), Invoice("90", discount="0.30"), Invoice("17", shipping="6")]),
    ([Invoice("9", quantity=4, discount="0.20", shipping="2")],
     [Invoice("14", quantity=2, discount="0.15", shipping="5")]),
]
META_HELD_OUT = [
    ([Invoice("6.25", quantity=6), Invoice("80", discount="0.40"), Invoice("19", shipping="7.25")],
     [Invoice("22", quantity=3), Invoice("35", discount="0.20"), Invoice("29", shipping="1.50"),
      Invoice("16", quantity=4, discount="0.25", shipping="8")]),
    ([Invoice("30", quantity=2, discount="0.30", shipping="4")],
     [Invoice("23", quantity=7, discount="0.10", shipping="2")]),
]


def evaluate_policy(policy, episodes):
    records = []
    for training, validation in episodes:
        adapted, adaptation_log = improve(
            INITIAL_AGENT, lambda state: evaluate_task(state, training), policy, client)
        records.append({"adaptation_log": adaptation_log,
                        "adaptation_feedback": evaluate_task(adapted, training),
                        "validation_score": evaluate_task(adapted, validation)["score"]})
    return {"score": sum(r["validation_score"] for r in records) / len(records),
            "episodes": records}


meta_evaluator = lambda policy: evaluate_policy(policy, META_DEVELOPMENT)
learned_policy, meta_log = improve(INITIAL_POLICY, meta_evaluator, INITIAL_POLICY, client)
(ARTIFACT_DIR / "policy.json").write_text(json.dumps(learned_policy, indent=2))
learned_policy = json.loads((ARTIFACT_DIR / "policy.json").read_text())
print("Policy before:", INITIAL_POLICY["parameters"])
print("Policy after: ", learned_policy["parameters"])
print("Meta-development score:", meta_log["baseline_score"], "→", meta_log["final_score"])
print("Accepted:", any(step["accepted"] for step in meta_log["steps"]))


Policy before: {'max_fields_per_candidate': 1, 'proposal_instruction': 'Inspect failures and propose changes that improve the measured objective. Use execution evidence. Consider the candidate evaluation budget.'}
Policy after:  {'max_fields_per_candidate': 3, 'proposal_instruction': 'Inspect failures and propose changes that improve the measured objective. Use execution evidence. Consider the candidate evaluation budget.'}
Meta-development score: 0.16666666666666666 → 1.0
Accepted: True


## Reuse the changed policy, then evaluate on new episodes

Compare both policies on identical fresh agents and adaptation examples. These
episodes were not supplied to meta-proposal generation. This is a tiny same-domain
holdout, not evidence of broad transfer. Cache exact repeated model requests to
avoid duplicate calls; the run is a paired cached demonstration, not independent
sampling. Temperature zero is not a guarantee of reproducibility.

Candidate budgets match, but total compute and changed-field counts do not.
Meta-search is extra work. Results are reported as observed, with no assertion
that Gemini must improve. Invalid or non-improving edits cannot be promoted.


In [6]:
agent_with_new_policy, new_agent_log = improve(INITIAL_AGENT, task_evaluator, learned_policy, client)
old_fresh_score = evaluate_policy(INITIAL_POLICY, META_HELD_OUT)["score"]
new_fresh_score = evaluate_policy(learned_policy, META_HELD_OUT)["score"]
next_policy, next_meta_log = improve(learned_policy, meta_evaluator, learned_policy, client)
summary = {
    "backend": BACKEND, "model": MODEL if BACKEND == "gemini" else None,
    "refinement": refinement_result,
    "agent_held_out_before": evaluate_task(INITIAL_AGENT, HELD_OUT)["score"],
    "agent_held_out_old_policy": evaluate_task(learned_agent, HELD_OUT)["score"],
    "agent_held_out_new_policy": evaluate_task(agent_with_new_policy, HELD_OUT)["score"],
    "fresh_episode_score_old_policy": old_fresh_score,
    "fresh_episode_score_new_policy": new_fresh_score,
    "retained_policy": learned_policy["parameters"],
    "next_recursive_round_policy": next_policy["parameters"],
    "successful_api_calls": len(client.calls) if BACKEND == "gemini" else 0,
    "total_tokens": sum(c.get("usage", {}).get("totalTokenCount", 0) for c in client.calls),
    "reported_call_seconds": round(sum(c.get("seconds", 0) for c in client.calls), 3),
}
(ARTIFACT_DIR / "result.json").write_text(json.dumps(summary, indent=2))
(ARTIFACT_DIR / "audit.json").write_text(json.dumps({
    "summary": summary, "agent": agent_log, "meta": meta_log,
    "new_agent": new_agent_log, "next_meta": next_meta_log, "calls": client.calls}, indent=2))
print(json.dumps(summary, indent=2))
print("Saved artifacts under", str(ARTIFACT_DIR.resolve()))


{
  "backend": "gemini",
  "model": "gemini-3.8-flash",
  "refinement": {
    "first_answer": "100.00",
    "refined_answer": "80.00",
    "next_request_answer": "100.00",
    "passed": true
  },
  "agent_held_out_before": 0.0,
  "agent_held_out_old_policy": 0.25,
  "agent_held_out_new_policy": 1.0,
  "fresh_episode_score_old_policy": 0.125,
  "fresh_episode_score_new_policy": 1.0,
  "retained_policy": {
    "max_fields_per_candidate": 3,
    "proposal_instruction": "Inspect failures and propose changes that improve the measured objective. Use execution evidence. Consider the candidate evaluation budget."
  },
  "next_recursive_round_policy": {
    "max_fields_per_candidate": 3,
    "proposal_instruction": "Inspect failures and propose changes that improve the measured objective. Use execution evidence. Consider the candidate evaluation budget."
  },
  "successful_api_calls": 12,
  "total_tokens": 12055,
  "reported_call_seconds": 60.139
}
Saved artifacts under ./gemini_artifacts


## What remains fixed, and what this demonstrates

The target is invoice accuracy; free parameters are task rules and improvement
policy fields. Constraints include the parameter schemas, one-candidate adaptation
budget, global API-call cap, fixed interpreter, and fixed external evaluator.

| Level | Changed object | Retained? |
|---|---|---|
| Output refinement | Temporary rules for one invoice answer | No |
| Agent self-improvement | Task rules | Yes |
| Recursive self-improvement | Policy governing subsequent improvement proposals | Yes |

Editable optimizer parameters enable recursion. A higher score on fresh adaptation
episodes is evidence of benefit in this bounded setting. One useful policy change
does not establish sustained acceleration; a later round can reach a ceiling.

There is still a task-agent/improver separation: Python executes tasks; Gemini
proposes edits. It is a role boundary, not an immutable modification boundary.
The improver's configurable procedure is itself an optimization target.
